# EXP-01 — Jupyter Experiment Harness Smoke & Colab Preparation

Development-only harness validation. Tests mode-aware loader, feature integrity, skeleton output generation, mode-separated result writing, and Colab adapter. Run via Restart Kernel → Run All.

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'notebooks/experiment_harness.py').is_file(), 'Mở notebook từ root repo hoặc notebooks/'
sys.path.insert(0, str(root / 'notebooks'))
sys.path.insert(0, str(root / 'tests'))

from experiment_harness import (
    load_feature_bundle,
    build_membership_rows,
    build_sequence_summary_row,
    write_experiment_result,
    ColabPathAdapter,
    MODE_DEVELOPMENT,
    MODE_OFFICIAL,
)
print('EXP-01 harness imported successfully.')

## 1. Load Development Pilot Bundle

Loads the DAT-02 pilot bundle. Validates checksums, schema, finite constraints, and self-vectors before fitting.

In [ ]:
import tempfile
from test_experiment_harness import create_pilot_fixture

bundle_path = os.environ.get('PILOT_BUNDLE', '')
_temp_dir = None
if bundle_path and Path(bundle_path).is_dir():
    pilot_dir = Path(bundle_path).resolve()
else:
    # Use self-contained synthetic pilot fixture if PILOT_BUNDLE is not provided
    _temp_dir = tempfile.TemporaryDirectory()
    pilot_dir = Path(_temp_dir.name)
    create_pilot_fixture(pilot_dir)

bundle = load_feature_bundle(pilot_dir, mode=MODE_DEVELOPMENT)
assert bundle['mode'] == MODE_DEVELOPMENT
assert bundle['is_official'] is False
assert bundle['manifest']['allowed_usage'] == 'DEVELOPMENT_ONLY'
assert bundle['manifest']['research_ready'] is False

features = bundle['features_table']
print('Loaded Pilot ID:', bundle['bundle_id'])
print('Feature count:', features.num_rows)
print('Column names:', features.column_names)
assert features.num_rows > 0

## 2. Generate Output Skeleton & Mode-Separated Writing

Builds valid skeletal memberships and sequence summary conforming strictly to result-bundle-v1. Verifies that development mode writes `pilot_result_manifest.json` and NEVER produces `_SUCCESS.json`.

In [ ]:
feature_rows = features.to_pylist()
mainshock_id = bundle['manifest']['selected_mainshock_ids'][0]

mem_rows = build_membership_rows(
    experiment_run_id="pilot_smoke_run",
    dataset_id=bundle['source_dataset_id'],
    mainshock_event_id=mainshock_id,
    candidate_records=feature_rows,
    algorithm_name="WINDOW",
    model_config_version="window_baseline_v1",
)

summary_row = build_sequence_summary_row(
    experiment_run_id="pilot_smoke_run",
    dataset_id=bundle['source_dataset_id'],
    mainshock_event_id=mainshock_id,
    candidate_count=len(feature_rows),
    algorithm_name="WINDOW",
    model_config_version="window_baseline_v1",
    membership_rows=mem_rows,
    window_result_status="SUCCESS",
)

with tempfile.TemporaryDirectory() as out_temp:
    out_dir = Path(out_temp)
    result = write_experiment_result(
        output_dir=out_dir,
        bundle=bundle,
        algorithm_name="WINDOW",
        algorithm_version="1.0.0",
        model_config_version="window_baseline_v1",
        model_config={"time_window_hours": 720.0},
        memberships=mem_rows,
        summaries=[summary_row],
        metrics={"runtime_seconds": 0.05},
        mode=MODE_DEVELOPMENT,
    )
    assert result['has_success_marker'] is False
    assert not (out_dir / '_SUCCESS.json').exists(), 'PILOT RUN MUST NEVER EMIT _SUCCESS.json'
    assert (out_dir / 'pilot_result_manifest.json').exists()
    assert (out_dir / 'memberships.parquet').exists()
    assert (out_dir / 'sequence_summary.parquet').exists()
    print('Development result successfully written without _SUCCESS marker.')

if _temp_dir is not None:
    _temp_dir.cleanup()

## 3. Colab Path Adapter & Migration Gate

Verifies that the Colab path adapter resolves parameterized paths without hardcoding credentials or absolute home paths, and enforces that migration to cloud is blocked if the local smoke gate is not PASSED.

In [ ]:
adapter = ColabPathAdapter()

# 1. Parameterized path resolution
local_in, local_out = adapter.resolve_paths(
    input_bundle="staging/ml-pilot",
    output_dir="staging/ml-output",
    is_colab=False
)
colab_in, colab_out = adapter.resolve_paths(
    input_bundle="japan-earthquake/pilot",
    output_dir="japan-earthquake/output",
    drive_mount="/content/drive/MyDrive",
    is_colab=True
)
print('Local paths resolved:', local_in.name, local_out.name)
print('Colab paths resolved:', colab_in.as_posix(), colab_out.as_posix())

# 2. Local smoke gate validation with mock report
with tempfile.TemporaryDirectory() as gate_temp:
    mock_report = Path(gate_temp) / "local-smoke-report.json"
    mock_report.write_text('{"local_smoke_status": "PASSED", "colab_migration_allowed": true}')
    gate_check = adapter.verify_local_smoke_gate(mock_report)
    assert gate_check["local_smoke_status"] == "PASSED"

print('EXP-01 HARNESS SMOKE PASSED: All requirements verified without secrets or credential dependencies.')